# Preprocessing data and running Beast inference on Pertussis data set in France between 2010 and 2024 under a constant-size coalescent and a skygrid prior

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import dendropy

from evaluation_helpers import *

## 1. Prepare metadata

In [47]:
meta_df = pd.read_csv("data/pertussis_france/Supplementary_information_isolates_CMI_Bouchez_2025.csv")
meta_df = meta_df[(meta_df["Year"] >= 2010) & (meta_df["Country"] == "France")]
meta_df["Isolate"].to_csv("data/pertussis_france/subsampled_ids.txt", index=False, header=False)  

def to_decimal_year(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # Case 1: year only, e.g. 2011
    if x.isdigit() and len(x) == 4:
        return float(x)

    # Case 2: full date, e.g. 22.03.2010
    date = pd.to_datetime(x, format="%d.%m.%Y", errors="coerce")

    if pd.isna(date):
        return np.nan

    year_start = pd.Timestamp(year=date.year, month=1, day=1)
    next_year_start = pd.Timestamp(year=date.year + 1, month=1, day=1)

    decimal_year = date.year + (date - year_start) / (next_year_start - year_start)

    return decimal_year

# create a file that maps taxon labels to dates
full_metadata_df = pd.read_csv("/Users/mariebecker/Documents/Uni/ETH/RotationStadler/confidential/Metadata_isolates_France_20241217_NL.csv")
full_metadata_df = full_metadata_df[(full_metadata_df['Collection_time'] >= 2010) & (full_metadata_df['Country'] == 'France')]
full_metadata_df = full_metadata_df[['Isolate', 'Accession_number', 'Collection_time', 'Country', 'Prelevement_date', 'Precision_plevement_date']]
full_metadata_df["Id"] = full_metadata_df["Accession_number"].fillna(full_metadata_df["Isolate"])
full_metadata_df["decimal_year"] = full_metadata_df["Prelevement_date"].apply(to_decimal_year)
#full_metadata_df["Id"].to_csv("data/pertussis_france/full_subsampled_ids.txt", index=False, header=False)
#full_metadata_df.to_csv("data/pertussis_france/full_metadata.csv", index=False)

# Create matching key in meta_df
# - if meta Isolate starts with E: match against Accession_number
# - if meta Isolate starts with F: match part before "_" against Isolate
meta_df["match_key"] = np.where(
    meta_df["Isolate"].str.startswith("E"),
    "ACC_" + meta_df["Isolate"],
    np.where(
        meta_df["Isolate"].str.startswith("F"),
        "ISO_" + meta_df["Isolate"].str.split("_").str[0].str.split("-").str[0],
        np.nan
    )
)

# Create matching keys in full_metadata_df
full_iso = full_metadata_df[["Isolate", "decimal_year"]].copy()
full_iso["match_key"] = "ISO_" + full_iso["Isolate"]

full_acc = full_metadata_df[["Accession_number", "decimal_year"]].copy()
full_acc["match_key"] = "ACC_" + full_acc["Accession_number"]

# Combine possible matches from full_metadata_df
full_lookup = pd.concat(
    [
        full_iso[["match_key", "decimal_year"]],
        full_acc[["match_key", "decimal_year"]],
    ],
    ignore_index=True
)

# Remove missing / invalid keys
full_lookup = full_lookup.dropna(subset=["match_key", "decimal_year"])

# Merge accurate decimal years into meta_df
meta_df = meta_df.merge(
    full_lookup,
    on="match_key",
    how="left",
    suffixes=("", "_accurate")
)

meta_df.to_csv("data/pertussis_france/full_metadata.csv", index=False)
meta_df[["Isolate", "decimal_year"]].to_csv(
    "data/pertussis_france/taxon_dates.txt",
    sep="\t",
    index=False,
    header=False
)

meta_df.head()

# subsampled fasta by these ids
# python subsample_fasta.py --fasta ../data/pertussis_france/MSA_pertussis_2024.fasta --ids ../data/pertussis_france/subsampled_ids.txt --out ../data/pertussis_france/MSA_pertussis_2010_2024_France.fasta
# kept 1133 / 1588 sequences

,Isolate,Year,Country,ptxP,ptxA,fim2,fim3,23S_muts,Region,ptxP3_MRBP,Introductions,match_key,decimal_year
0,FR7673_S80,2024.0,France,ptxP3,ptxA1,fim2-1,fim3-1,23S-3,IDF,ptxP3-MRBP,I1,ISO_FR7673,2024.368852
1,FR8322_S165,2024.0,France,ptxP3,ptxA1,fim2-1,fim3-1,23S-3,GES,ptxP3-MRBP,I2,ISO_FR8322,2024.603825
2,FR8344_S169,2024.0,France,ptxP3,ptxA1,fim2-1,fim3-1,23S-3,GES,ptxP3-MRBP,I2,ISO_FR8344,2024.609290
3,FR8581_S145,2024.0,France,ptxP3,ptxA1,fim2-1,fim3-1,23S-3,GES,ptxP3-MRBP,I2,ISO_FR8581,2024.734973
4,FR8573_S140,2024.0,France,ptxP3,ptxA1,fim2-1,fim3-1,23S-3,ARA,ptxP3-MRBP,I2,ISO_FR8573,2024.000000


## 2. Prepare fasta files
- Subsample fasta file to only include samples after 2010 (`scripts/subsample_fasta.py`)
- run gubbins to remove recombination sites (`scripts/run_gubbins.sh`)
- transform gubbins `.tre` output tree into nexus format, which is needed as input for Beauty
- run snp-sites to only keep SNPs
- count invariant bases to add base counts to xml file (`scripts/count_invariant_bases.py`)

Subsampled fasta by these filtered ids:

```
python subsample_fasta.py --fasta ../data/pertussis_france/MSA_pertussis_2024.fasta --ids ../data/pertussis_france/subsampled_ids.txt --out ../data/pertussis_france/MSA_pertussis_2010_2024_France.fasta
```

kept 1133 / 1588 sequences

In [ ]:
# Convert the Gubbins final tree from Newick to NEXUS, restoring the underscores that dendropy turns into spaces and removing quotes so tip labels match the alignment exactly.

tree = dendropy.Tree.get(path="./data/pertussis_france/pertussis_france.final_tree.tre", schema="newick")                                                                                                                                                   
for leaf in tree.leaf_node_iter():                                                                                                                                                                                                  
    leaf.taxon.label = leaf.taxon.label.replace(" ", "_")
                                                                                                                                                                                                                                    
tree.write(
    path="./data/pertussis_france/pertussis_france.final_tree.nexus",                                                                                                                                                                                       
    schema="nexus",
    unquoted_underscores=True,
)

with open("./data/pertussis_france/pertussis_france.final_tree.nexus", "r") as f:
    content = f.read()                                                                                                                                                                                                              
content = content.replace("'", "")
with open("./data/pertussis_france/pertussis_france.final_tree.nexus", "w") as f:                                                                                                                                                   
    f.write(content)  

Run snp sites on filtered dataset:
```
snp-sites -o pertussis_france.filtered_polymorphic_sites.snps.fasta pertussis_france.filtered_polymorphic_sites.fasta
```

Get counts of non snp-sites:
```
conda activate beast-tools
sbatch --job-name=count_bases --output=logs/count_bases_%j.out --error=logs/count_bases_%j.err --time=1:00:00 --mem-per-cpu=20G --wrap="python scripts/count_invariant_bases.py --full data/pertussis_france/MSA_pertussis_2010_2024_France.fasta"```

````
Output:
- Sequences        : 1133
- Alignment length : 4086189
- SNP positions    : 3288
- Invariant positions: 4082901
- Base counts at invariant positions:
  - A         :      609,313
  - C         :    1,282,107
  - T         :      608,839
  - G         :    1,291,439
  - N only    :      291,203
  - Gap only  :            0
  - N+Gap mix :            0
  - Total     :    4,082,901  (== invariant positions: 4,082,901)

## 3. Run BEAST
We ran each BEAST analysis (skygrid and constant-size coalescent) with three different random seeds. Because of the 120-hour job limit on the cluster, each run was split into three consecutive 120-hour jobs. Each job resumed from the checkpoint saved by the previous one, and was submitted with a dependency on that job, so it only started once the previous job had ended. The run script supports up to two continuations: it checks which checkpoint files already exist and resumes from the most recent one. For example:
```
sbatch --dependency=afterany:274544 --export=ALL,SEED=47 scripts/run_beast_skygrid.sh
sbatch --dependency=afterany:274555 --export=ALL,SEED=48 scripts/run_beast_skygrid.sh
sbatch --dependency=afterany:274578 --export=ALL,SEED=49 scripts/run_beast_skygrid.sh

sbatch --dependency=afterany:274600 --export=ALL,SEED=47 scripts/run_beast_constcoal.sh
sbatch --dependency=afterany:274613 --export=ALL,SEED=48 scripts/run_beast_constcoal.sh
sbatch --dependency=afterany:274622 --export=ALL,SEED=49 scripts/run_beast_constcoal.sh
```
